# Pipeline de Auditoria de Credenciamento
Este notebook orquestra as 5 fases do pipeline analítico. Ele foi adaptado para permitir a execução passo a passo, facilitando a inspeção dos dados (DataFrames) em cada etapa do funil antes de gerar o relatório executivo.

### Configuração de Ambiente e Importações
Inicializa os módulos de log, carrega as chaves de API (.env) e importa os scripts de cada fase.

In [1]:
import logging
import inspect
import pandas as pd
from dotenv import load_dotenv
import nest_asyncio

# Permite rodar rotinas assíncronas (como o Gemini) dentro do Jupyter Notebook
nest_asyncio.apply()

# Carrega variáveis de ambiente (GEMINI_API_KEY)
load_dotenv()

# Configuração global de log do orquestrador
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')

# Importações dos módulos locais do pipeline
from fase_1 import executar_fase_1
from fase_2 import executar_fase_2
from fase_3 import executar_fase_3
from fase_4 import executar_fase_4
from fase_5 import executar_fase_5

# Arquivo que será ingerido pelo pipeline
ARQUIVO_FONTE = '../Base dedados de Licitação em 17-09.xlsx' # Mude para o caminho correto caso necessário

### FASE 1: Ingestão e Higienização
Carrega os dados brutos, valida a chave primária e padroniza as colunas de texto e valores financeiros.

In [2]:
logging.info('Iniciando Fase 1: Ingestão e Higienização...')

df_analytical, df_raw = executar_fase_1(ARQUIVO_FONTE)

print(f'\nTotal de registros únicos: {len(df_analytical)}')
df_analytical.head()

[INFO] Iniciando Fase 1: Ingestão e Higienização...
[INFO] Registros iniciais: 46476 | Protocolos únicos: 23386
[INFO] Fase 1 concluída com sucesso. Base analítica consolidada e higienizada.



Total de registros únicos: 23386


,Protocolo,Categoria,Subcategoria,Setor Atual,No.Licitação,Julgado?,Município,Jurisdicionado,Modalidade,Tipo do Objeto,...,R6_REGRA,D1_ENQUADRAMENTO,D2_EXECUCAO,D3_PRECIFICACAO,D4_DISTRIBUICAO,D5_RECORRENCIA,QTD_DIMENSOES,PRIORIDADE_QUALITATIVA,NECESSITA_REVISAO_MANUAL,JUSTIFICATIVA_TRIAGEM
0,Doc. 00011/26,Licitações e Contratos,Licitações,CARTÓRIO DIAFI,00001/2025,Não,PEDRA LAVRADA,CAMARA MUNICIPAL DE PEDRA LAVRADA,DISPENSA (LEI No 14.133/2021),OBRAS E SERVICOS DE ENGENHARIA,...,<NA>,False,False,False,False,False,0,Q4 — Monitoramento,False,<NA>
1,Doc. 00016/26,Licitações e Contratos,Licitações,CARTÓRIO DIAFI,00019/2025,Não,CACIMBA DE AREIA,PREFEITURA MUNICIPAL DE CACIMBA DE AREIA,INEXIGIBILIDADE (LEI No 14.133/2021),COMPRAS E SERVICOS,...,<NA>,False,False,False,False,False,0,Q4 — Monitoramento,False,<NA>
2,Doc. 00018/26,Licitações e Contratos,Licitações,CARTÓRIO DIAFI,00014/2025,Não,LAGOA,CAMARA MUNICIPAL DE LAGOA,DISPENSA (LEI No 14.133/2021),COMPRAS E SERVICOS,...,<NA>,False,False,False,False,False,0,Q4 — Monitoramento,False,<NA>
3,Doc. 00020/26,Licitações e Contratos,Licitações,CARTÓRIO DIAFI,00014/2025,Não,JACARAU,CAMARA MUNICIPAL DE JACARAU,DISPENSA (LEI No 14.133/2021),COMPRAS E SERVICOS,...,<NA>,False,False,False,False,False,0,Q4 — Monitoramento,False,<NA>
4,Doc. 00022/26,Licitações e Contratos,Licitações,CARTÓRIO DIAFI,00009/2025,Não,CONDADO,PREFEITURA MUNICIPAL DE CONDADO,ADESAO A ATA DE REGISTRO DE PRECOS (LEI No 14....,OBRAS E SERVICOS DE ENGENHARIA,...,<NA>,False,False,False,False,False,0,Q4 — Monitoramento,False,<NA>


In [3]:
df_raw.columns

Index(['Protocolo', 'Categoria', 'Subcategoria', 'Setor Atual', 'No.Licitação',
       'Julgado?', 'Município', 'Jurisdicionado', 'Modalidade',
       'Tipo do Objeto', 'Objeto', 'Estágio', 'Fracassada?',
       'CPF/CNPJ Proponente', 'Proponente', 'Situação Proposta',
       'Ano Homologação', 'Valor Estimado', 'Valor Proposta',
       'Valor Licitação', 'Valor Contrato', 'Valor Aditivo'],
      dtype='str')

### FASE 2: Triagem Determinística e Vetorial
Filtra os 'lixos óbvios', identifica menções incidentais e isola a 'zona cinzenta' (busca léxica e similaridade semântica local) para a fila da IA.

In [4]:
logging.info('Iniciando Fase 2: Triagem Determinística e Peneira Vetorial...')

df_analytical, df_queue = executar_fase_2(df_analytical)

print(f'\nTamanho da fila separada para avaliação do LLM: {len(df_queue)}')
df_queue.head()

[INFO] Iniciando Fase 2: Triagem Determinística e Peneira Vetorial...
[INFO] Filtro Direto Grau A (Vínculo a Protocolo de Credenciamento): 292 registros classificados diretamente como INCLUÍDO (Divergências: 4).
[INFO] Filtro Direto Grau A (Agricultura Familiar / PNAE): 226 registros classificados diretamente como INCLUÍDO (Divergências: 3).
[INFO] Processando 15179 registros na Peneira Fina Semântica...
[INFO] Use pytorch device_name: cpu
[INFO] Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
c:\Users\akendall\Downloads\analise-licitacoes\venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Batches: 100%|██████████| 475/47


Tamanho da fila separada para avaliação do LLM: 15185


,Protocolo,Modalidade,Objeto,GI - Regime Jurídico Informado,Valor,R1_STATUS,R1_EVIDENCIA,R2_STATUS,R2_EVIDENCIA
1,Doc. 00016/26,INEXIGIBILIDADE (LEI No 14.133/2021),APRESENTACAO DE UM SHOW ARTISTICO DA DUPLA EDY...,Lei 14.133/2021,100000.0,NÃO IDENTIFICADO,<NA>,NÃO IDENTIFICADO,<NA>
2,Doc. 00018/26,DISPENSA (LEI No 14.133/2021),"CONTRATACAO DE EMPRESA PARA PLANEJAMENTO, ELAB...",Lei 14.133/2021,21250.0,NÃO IDENTIFICADO,<NA>,NÃO IDENTIFICADO,<NA>
3,Doc. 00020/26,DISPENSA (LEI No 14.133/2021),"CONTRATACAO DE SERVICOS DE DIGITALIZACAO, CATA...",Lei 14.133/2021,48000.0,NÃO IDENTIFICADO,<NA>,NÃO IDENTIFICADO,<NA>
8,Doc. 00030/26,DISPENSA (LEI No 14.133/2021),CONTRATACAO DE SERVICOS DE PROTOCOLO ELETRONIC...,Lei 14.133/2021,30000.0,NÃO IDENTIFICADO,<NA>,NÃO IDENTIFICADO,<NA>
9,Doc. 00033/26,PREGAO (LEI No 14.133/2021),CONTRATACAO DE EMPRESA ESPECIALIZADA PARA EXEC...,Lei 14.133/2021,471670.0,NÃO IDENTIFICADO,<NA>,NÃO IDENTIFICADO,<NA>


In [5]:
#df_analytical.to_csv('df_analytical_fase2.csv', index=False)
#df_queue.to_csv('df_queue_fase2.csv', index=False)

### FASE 3: Análise Semântica (LLM - Gemini)
Envia a fila isolada na fase anterior para avaliação de mérito utilizando o modelo Gemini (assíncrono), extraindo o raciocínio e possíveis divergências cadastrais da Prefeitura.

In [6]:
#df_analytical_sample = pd.read_excel('df_analytical_sample.xlsx')
#df_queue_sample = pd.read_excel('df_queue_sample.xlsx')

Usando só uma sample do df completo:

In [7]:

df_queue_sample = df_queue.sample(n=450, random_state=42)
df_analytical_sample = df_analytical[df_analytical['Protocolo'].isin(df_queue_sample['Protocolo'])].copy()

df_queue_sample.to_excel('df_queue_sample.xlsx', index=False)
df_analytical_sample.to_excel('df_analytical_sample.xlsx', index=False)

In [8]:
logging.info('Iniciando Fase 3: Julgamento Semântico via LLM (Assíncrono)...')

# Como o Jupyter tem seu próprio event loop, podemos usar wait diretamente na célula
if inspect.iscoroutinefunction(executar_fase_3):
    df_analytical_fase3 = await executar_fase_3(df_analytical_sample, df_queue_sample)
else:
    df_analytical_fase3 = executar_fase_3(df_analytical_sample, df_queue_sample, modo_gratuito=True)

# Mostra o balanço de como os dados foram classificados
display(df_analytical_fase3[['GI - Resultado Etapa 1', 'GI - Grau de Evidência']].value_counts())

[INFO] Iniciando Fase 3: Julgamento Semântico via LLM (Assíncrono)...
[INFO] Realizando merge estruturado de 450 registros no DataFrame analítico...
[INFO] Fase 3 concluída com sucesso. Base analítica enriquecida com insights semânticos.



📊 PAINEL DE CONTROLE — FASE 3 (JULGAMENTO SEMÂNTICO VIA LLM)
 • Total de protocolos na fila da IA : 450
 • Já processados (Checkpoint/Mem.)  : 450 (100.0%)
 • Pendentes para concluir a base    : 0
 • Status                            : ✅ 100% CONCLUÍDO! Nenhuma chamada necessária.



GI - Resultado Etapa 1  GI - Grau de Evidência      
EXCLUÍDO                GRAU C — SEM EVIDÊNCIA          286
INCLUÍDO                GRAU A — EVIDÊNCIA DIRETA       112
                        GRAU B — SINTOMÁTICO             50
VALIDAÇÃO PENDENTE      GRAU D — TEXTO VAGO/GENÉRICO      2
Name: count, dtype: int64

In [9]:
#df_analytical_fase3.to_csv('df_analytical_fase3.csv', index=False)
#df_analytical_fase3.to_excel('df_analytical_fase3.xlsx', index=False)

### FASE 4: Corroboração Interna e Testes de Integridade
Aplica heurísticas para identificar clusters de mercado fluido (Promoção para Grau A) e executa os testes vitais de consistência dos dados.

In [ ]:
# ROTINA DIÁRIA (500 LINHAS/DIA DO DATASET COMPLETO NO MODO GRATUITO ATÉ TERÇA-FEIRA):
# Usa o cache local da Fase 2 (ou gera na 1ª vez), processa mais 500 protocolos pendentes
# do dataset completo (df_queue), salva em checkpoint_fase_3.jsonl e atualiza o Excel acumulado.
from rodar_lote_diario import executar_lote_do_dia

df_analytical_fase3 = executar_lote_do_dia(limite_diario=500, gerar_relatorio_parcial=True)

In [10]:
#df_analytical = pd.read_excel('df_analytical_fase3.xlsx')

In [11]:
df_analytical = df_analytical_fase3.copy()

In [12]:
logging.info('Iniciando Fase 4: Corroboração Interna e Asserções de Qualidade...')

df_analytical = executar_fase_4(df_analytical)

# Mostra o resumo da Prioridade Qualitativa (Q1 a Q4) do Motor R0–R6
print('\nResumo da Triagem Qualitativa (Q1 a Q4):')
display(df_analytical['PRIORIDADE_QUALITATIVA'].value_counts())

# Mostra as contratações irmãs promovidas (se houver)
mask_promovidos = df_analytical['GI - Corroboração Utilizada'].notna()
display(df_analytical[mask_promovidos][['Protocolo', 'Município', 'Modalidade', 'GI - Corroboração Utilizada', 'GI - Grau de Evidência', 'PRIORIDADE_QUALITATIVA']])

[INFO] Iniciando Fase 4: Corroboração Interna e Asserções de Qualidade...
[INFO] 71 registros promovidos a 'GRAU A — EVIDÊNCIA DIRETA' via Corroboração Interna.
[INFO] ✅ Todos os testes de qualidade (V1 a V6) passaram com sucesso.
[INFO] Fase 4 concluída. Dados estruturados, corroborados e matriz R0–R6 consolidada.



Resumo da Triagem Qualitativa (Q1 a Q4):


PRIORIDADE_QUALITATIVA
Q4 — Monitoramento    342
Q2 — Alta              64
Q3 — Média             40
Q1 — Muito alta         4
Name: count, dtype: int64

,Protocolo,Município,Modalidade,GI - Corroboração Utilizada,GI - Grau de Evidência,PRIORIDADE_QUALITATIVA
0,Doc. 00844/26,ITAPOROROCA,INEXIGIBILIDADE (LEI No 14.133/2021),CORROBORAÇÃO INTERNA (CONTRATAÇÕES IRMÃS),GRAU A — EVIDÊNCIA DIRETA,Q4 — Monitoramento
9,Doc. 03445/26,JOAO PESSOA,INEXIGIBILIDADE (LEI No 14.133/2021),CORROBORAÇÃO INTERNA (CONTRATAÇÕES IRMÃS),GRAU A — EVIDÊNCIA DIRETA,Q4 — Monitoramento
25,Doc. 09504/26,JACARAU,INEXIGIBILIDADE (LEI No 14.133/2021),CORROBORAÇÃO INTERNA (CONTRATAÇÕES IRMÃS),GRAU A — EVIDÊNCIA DIRETA,Q4 — Monitoramento
46,Doc. 104509/26,ALAGOA GRANDE,INEXIGIBILIDADE (LEI No 14.133/2021),CORROBORAÇÃO INTERNA (CONTRATAÇÕES IRMÃS),GRAU A — EVIDÊNCIA DIRETA,Q4 — Monitoramento
50,Doc. 10578/26,IBIARA,INEXIGIBILIDADE (LEI No 14.133/2021),CORROBORAÇÃO INTERNA (CONTRATAÇÕES IRMÃS),GRAU A — EVIDÊNCIA DIRETA,Q4 — Monitoramento
...,...,...,...,...,...,...
437,Doc. 95236/26,JOAO PESSOA,INEXIGIBILIDADE (LEI No 14.133/2021),CORROBORAÇÃO INTERNA (CONTRATAÇÕES IRMÃS),GRAU A — EVIDÊNCIA DIRETA,Q4 — Monitoramento
438,Doc. 95249/26,JOAO PESSOA,INEXIGIBILIDADE (LEI No 14.133/2021),CORROBORAÇÃO INTERNA (CONTRATAÇÕES IRMÃS),GRAU A — EVIDÊNCIA DIRETA,Q4 — Monitoramento
443,Doc. 96807/26,BOA VISTA,DISPENSA (LEI No 14.133/2021),CORROBORAÇÃO INTERNA (CONTRATAÇÕES IRMÃS),GRAU A — EVIDÊNCIA DIRETA,Q2 — Alta
444,Doc. 97443/26,NOVA FLORESTA,INEXIGIBILIDADE (LEI No 14.133/2021),CORROBORAÇÃO INTERNA (CONTRATAÇÕES IRMÃS),GRAU A — EVIDÊNCIA DIRETA,Q4 — Monitoramento


### FASE 5: Geração do Entregável Executivo
Salva o arquivo final (.xlsx) contendo as formatações condicionais de alerta, as quebras de linha e o roteamento de abas.

In [13]:
logging.info('Iniciando Fase 5: Geração do Relatório Executivo Excel...')

nome_arquivo_excel = executar_fase_5(df_analytical, df_raw)

print(f'\n✅ Pipeline finalizado! Relatório pronto em: {nome_arquivo_excel}')

[INFO] Iniciando Fase 5: Geração do Relatório Executivo Excel...
[INFO] Iniciando a geração do entregável executivo: Auditoria_Credenciamento_Resultados_2026-10-27.xlsx
[INFO] Fase 5 concluída com sucesso. Entregável Excel salvo: 'Auditoria_Credenciamento_Resultados_2026-10-27.xlsx'.



✅ Pipeline finalizado! Relatório pronto em: Auditoria_Credenciamento_Resultados_2026-10-27.xlsx
